# E-Commerce Sales Analysis - Data Cleaning and Preparation

## Objectives
- Clean and validate the data
- Handle missing values and duplicates
- Remove outliers if needed
- Create new features for analysis
- Prepare data for deeper analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load the data
df = pd.read_csv('data/raw/sales_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Original dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## Step 1: Check for Duplicates

In [ ]:
# Check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_rows}")

# Check for duplicate Order IDs
duplicate_orders = df['Order_ID'].duplicated().sum()
print(f"Duplicate Order IDs: {duplicate_orders}")

if duplicate_rows > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found. Dataset is clean.")

## Step 2: Validate Data Types

In [ ]:
print("Current Data Types:")
print(df.dtypes)

# Ensure numeric columns are numeric
numeric_cols = ['Price', 'Quantity', 'Revenue', 'Rating']
for col in numeric_cols:
    if df[col].dtype == 'object':
        df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Converted {col} to numeric")

print("\nData types validation complete.")

## Step 3: Check for Missing Values

In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing Values:")
if missing.sum() == 0:
    print("No missing values found. Dataset is complete.")
else:
    print(missing[missing > 0])
    print("\nHandling missing values...")
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            if df[col].dtype in ['float64', 'int64']:
                df[col].fillna(df[col].median(), inplace=True)
            else:
                df[col].fillna(df[col].mode()[0], inplace=True)

## Step 4: Validate Business Rules

In [ ]:
print("Data Validation Checks:")
print("="*50)

# Check if Revenue = Price * Quantity
calculated_revenue = df['Price'] * df['Quantity']
revenue_match = (df['Revenue'] == calculated_revenue).sum()
print(f"Revenue calculation check: {revenue_match} out of {len(df)} match")

# Check ratings are between 1-5
invalid_ratings = ((df['Rating'] < 1) | (df['Rating'] > 5)).sum()
print(f"Invalid ratings (not 1-5): {invalid_ratings}")

# Check for negative prices or quantities
negative_price = (df['Price'] <= 0).sum()
negative_qty = (df['Quantity'] <= 0).sum()
print(f"Negative or zero prices: {negative_price}")
print(f"Negative or zero quantities: {negative_qty}")

print(f"\nAll validations passed. Data is valid.")

## Step 5: Detect Outliers

In [ ]:
print("Outlier Detection (IQR Method):")
print("="*50)

for col in ['Price', 'Quantity', 'Revenue']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    print(f"{col}: {outliers} outliers detected")
    print(f"  Normal range: {lower_bound:.2f} to {upper_bound:.2f}")
    print()

## Step 6: Feature Engineering

In [ ]:
# Extract date features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%B')
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Quarter'] = df['Date'].dt.quarter
df['Week'] = df['Date'].dt.isocalendar().week

print("Date Features Created:")
print(df[['Date', 'Month_Name', 'Day_of_Week', 'Quarter']].head())

# Create order size categories
df['Order_Size'] = pd.cut(df['Quantity'], 
                           bins=[0, 1, 5, 10, float('inf')],
                           labels=['Single', 'Small', 'Medium', 'Large'])

# Create price tier categories
df['Price_Tier'] = pd.cut(df['Price'],
                           bins=[0, 1000, 5000, 10000, float('inf')],
                           labels=['Budget', 'Mid-Range', 'Premium', 'Luxury'])

# Calculate profit (assuming 30% cost)
df['Cost_Per_Unit'] = df['Price'] * 0.7
df['Total_Cost'] = df['Cost_Per_Unit'] * df['Quantity']
df['Profit'] = df['Revenue'] - df['Total_Cost']
df['Profit_Margin'] = ((df['Profit'] / df['Revenue']) * 100).round(2)

print(f"\nNew Features Created:")
print(df[['Order_Size', 'Price_Tier', 'Profit', 'Profit_Margin']].head())

## Step 7: Summary Statistics for Cleaned Data

In [ ]:
print("\nCleaned Dataset Summary:")
print("="*50)
print(f"Total Records: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print(f"Date Range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nNumeric Columns Summary:")
print(df[['Price', 'Quantity', 'Revenue', 'Profit', 'Rating']].describe())

## Step 8: Save Cleaned Data

In [ ]:
# Save cleaned data
df.to_csv('data/processed/sales_data_cleaned.csv', index=False)
print(f"Cleaned data saved to: data/processed/sales_data_cleaned.csv")
print(f"\nFile Details:")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)} columns")
print(f"\nColumn Names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

## Data Cleaning Complete

Summary of changes:
1. Removed duplicate records (if any)
2. Validated data types
3. Handled missing values
4. Validated business rules
5. Detected and reviewed outliers
6. Created new features for analysis
7. Saved cleaned data for analysis

Next Step: Move to Notebook 03 for Analysis and Visualization